# Introduction to Data Science and Systems 2026-2027
## Lab 6 (Week 8): Continuous Probabilities 
#### 
---
#### MSc Data Science, University of Glasgow


## What this lab is for

This lab notebook is for you to apply in practice some of the continuous probability concepts seen in class. 

It should take about **30 minutes**. **Lab 6 will count towards your course grade.**

## How the labs work (reminder)

* Questions are labelled **Task N**. Each one has an **answer cell** that starts out containing

      # YOUR CODE HERE
      raise NotImplementedError()

  Replace **both** of those lines with your answer. **Only edit cells that contain
  `# YOUR CODE HERE`** - changing anything else can corrupt the notebook and cost you marks.

* Note: Do not import additional python packages in your answers. The labs will never require it, 
and it will be penalised. 

* After a task there are one or more **test cells** (you cannot edit these). Run them to check your
  answer:
  * a green **&check; [n marks]** box means the test passed;
  * a red **&cross; Test failed** box means it did not.

* Some tests are **hidden**. Running a hidden test cell shows *nothing* now; it is only scored later.
  So "no output" from a test cell is normal and does **not** mean something is wrong.

* Before you finish, always use **Kernel &rarr; Restart Kernel and Run All Cells**, then check the
  marks summary at the bottom.

* When finished, submit your notebook to Moodle for marking. **Make sure to submit your work before the end of the session!**



Now import the packages the autograder needs. **Make sure you run this cell.**

In [ ]:
## Standard imports -- run this cell!
from __future__ import print_function, division
import os

# course grading utilities
%pip install -U --force-reinstall --no-cache https://github.com/pugeault/jhwutils/zipball/master
from jhwutils.checkarr import array_hash, check_hash, check_string, check_scalar
from jhwutils.matrices import print_matrix
import jhwutils.tick as tick
import jhwutils.image_audio as ia
from unittest.mock import patch

# numpy
import numpy as np

# plotting 
import matplotlib as mpl
import matplotlib.pyplot as plt

version_id = "v20262027a"

# for printouts
silent = False

# reset the visible marks counter
tick.reset_marks()

print("Everything imported OK")


In [ ]:
# This cell is used for grading purposes. Do not modify it.


---
## Task 0 &mdash; Tell us who you are

Set the two variables below:

* `student_id` &mdash; your 8-character student ID (for example `"1234567s"`).
* `student_typewritten_signature` &mdash; your full name, for example `"Ada Lovelace"`. Typing your
  name is your declaration that this notebook is your own work.

This is your first answer cell: replace the `# YOUR CODE HERE` / `raise NotImplementedError()` lines
with your answer, then run the test cell that follows.


In [ ]:
student_id = ""                     # your 8-char student id, e.g. "1234567s"
student_typewritten_signature = ""  # your full name, e.g. "Ada Lovelace"

# YOUR CODE HERE
raise NotImplementedError()


In [ ]:

with tick.marks(0):  # no credit for knowing your own id -- this is just a format check
    assert len(student_id) == 8

with tick.marks(0):
    assert len(student_typewritten_signature) > 0



## Hunt the submarine: continuous version


<img src="imgs/scorpion.jpg" width="50%">

The simple discrete grid model is easy to do computations with but not a very realistic model. A better world model is that locations are *continuous* in nature; for example points specified as longitude and latitude. To work with this model probabilistically, we need to use (multivariate, i.e. vector) **probability density functions** which map coordinates to *density*, not probability.



### Multivariate normal distribution
The PDF for the multivariate Normal distribution is defined as follows (for column vectors):

$$f_{{X}}(\mathbf{x})=\frac{1}{\sqrt{(2\pi)^d|\boldsymbol\Sigma|}}\operatorname{exp}\left( -\frac{1}{2} (\mathbf{x}-\boldsymbol\mu)^\top \boldsymbol\Sigma^{-1} (\mathbf{x}-\boldsymbol\mu)   \right)$$
where $\boldsymbol\mu$ is the mean (vector), $d$ is the dimension of $\mathbf{x}$ and $\boldsymbol\Sigma$ is the $d \times d$ covariance matrix.

### Continuous distributions
A continuous random variable can be represented as a class that has parameters and that provides two functions:

* `llik` the log-likelihood of an observation
* `sample` that returns a sample from a distribution

The multivariate normal PDF is implemented below using standard `scipy` implementations along with useful methods for drawing the density for debugging.

In [ ]:

from submarine import submarine_pmf, show_pmf

import scipy.stats
from jhwutils.ellipse import _cov_ellipse

class MultivariateNormal:
    
    def __init__(self, mu, sigma):
        # use the existing implementation in scipy.stats
        self.mu = mu
        self.sigma = sigma
        self.distribution = scipy.stats.multivariate_normal(self.mu, self.sigma)
    
    # log-likelihood of one observation
    def llik(self, x):
        return self.distribution.logpdf(x)
    
    # n random samples
    def sample(self, n):
        return self.distribution.rvs(n)        
        
    # render the distribution pdf as if it were a PMF
    def draw(self):            
        span = np.linspace(0,16,160)
        grid = np.array(np.meshgrid(span, span)).reshape(2, -1).T        
        pdf = self.distribution.pdf(grid)        
        show_pmf(pdf.reshape(160,160).T)                        

---
### Task 1
Instantiate a multivariate Normal distribution using the `MultivariateNormal` class defined above. Store it in `x`. E.g. `x = MultivariateNormal(mu=m, sigma=S)`

The PDF should have mean with coordinates (6,6), variance of one along both the latitude and longitude direction, and a covariance of 0.6.


In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

# to display the distribution on the map
x.draw()

In [ ]:
with tick.marks(1):
    assert check_hash(x.mu, ((2,), 42.0))
    

In [ ]:
with tick.marks(1):
    assert check_hash(x.sigma, ((2,2), 11.4))

---
### Task 2 - Fitting with direct computation

There are a collection of *observed* potential locations for the submarine from a search ship in `submarine_observations`. The problem is to fit the parameters of a PDF given these observations.

In [ ]:
# load the data from file
submarine_observations = np.loadtxt('data/submarine.txt')

print(submarine_observations.shape)

In [ ]:
# Sanity check that the data has been loaded successfully 

with tick.marks(0):
    assert check_hash(submarine_observations, ((200,2), 557639.8817902977)), "ERROR: Loaded data is wrong, check that you downloaded and unzipped file properly!"

Recall that the **arithmetic mean** is sum of sample values $\vec{x}_1, \vec{x}_2, \dots, \vec{x}_n$ divided by the number of values:
$$\hat{\boldsymbol\mu} = \frac{1}{n} \sum_{i=1}^n \vec{x}_i$$

The empirical covariance is computed by:

$${\boldsymbol{\hat \Sigma }} = \frac{1}{{n - 1}}\sum\limits_{i = 1}^n {{{\left( {{{\mathbf{x}}_i} - {\boldsymbol{\hat \mu }}} \right)}^T}\left( {{{\mathbf{x}}_i} - {\boldsymbol{\hat \mu }}} \right)} $$

Under the assumption that a reasonable model for the submarine location is a multivariate normal:

* directly estimate the parameters for a `MultivariateNormal` using standard numpy functions (which we have used in previous labs and several lectures) and store the result in `normal_mu_hat` and `normal_sigma_hat`.
* `llik_normal` the log-likelihood of the *entire* set of observations under this model (i.e. with these parameters)
* `samples_normal` 250 samples drawn from the multivariate normal model with these parameters
* `llik_misfit` how much smaller the log-likelihood of observations is compared to the samples *generated under the model* (i.e. `samples_normal`). This should be a negative number.

In [ ]:
# Set a seed - DO NOT MODIFY
np.random.seed(42)

# YOUR CODE HERE
raise NotImplementedError()

Once you have calculated your statistics for `normal_mu_hat` and `normal_sigma_hat`, you can run the cell below to draw samples from the resulting 
multivariate distribution and compare them to the observed data. 

In [ ]:
# This cell will display the sampled points and actual observations

x = MultivariateNormal(normal_mu_hat, normal_sigma_hat)

x.draw()
ax = plt.gca()
ax.scatter(samples_normal[:,0], samples_normal[:,1], c='r', marker='x', s=1, label="Samples")
ax.scatter(submarine_observations[:,0], submarine_observations[:,1], marker='o', color='w', s=2, label="Observations")
ax.set_xlim(0,16)
ax.set_ylim(0,16)
ax.legend()
ax.set_title(f"Log-likelihood = {llik_normal:.2f}")
plt.show()

In [ ]:
with tick.marks(1):
    assert check_hash(normal_mu_hat, ((2,), 48.304595497516765))

In [ ]:
with tick.marks(1):
    assert check_hash(normal_sigma_hat, ((2,2), 16.421940503564702))    

In [ ]:
# Hidden test checking the scalar llik_normal [1 mark]

In [ ]:
with tick.marks(2):
    assert samples_normal.shape == (250, 2)
    assert np.all(np.std(samples_normal, axis=0)>0.5)
    assert np.all(np.mean(samples_normal, axis=0) - np.array([8, 6]) < 4.0 * (np.std(samples_normal, axis=0)/np.sqrt(250)))  

In [ ]:
# Hidden test checking the scalar llik_misfit [1 mark]

### Part 2 - A cross search pattern

Scientists argue that a better model for the location of the submarine is a cross-shaped density that reflects two possible ocean currents in the Atlantic and is also easier to search by ship. The scientists have written a probabilistic model which gives the density of a specialised "cross distribution". This is a non-standard distribution that has four parameters: `ctr`, `span`, `leg` and `angle`.

<img src="imgs/ship_cross.png" width="50%">

The new density (formed with a combination of two multivariate normal distributions that are translated and transformed as we have seen in class) is denoted as $f^+_X(x)$ and defined in the Python class below: 

In [ ]:
class CrossDistribution:
    def __init__(self, ctr, span, leg, angle):       
        self.ctr = ctr
        leg = 1.0 + np.log(leg + 1.0)
        s, c = np.sin(np.radians(angle)), np.cos(np.radians(angle))
        r1 = np.array([[s / leg, c * leg], [c / leg, -s * leg]])
        r2 = np.array([[-c / leg, s * leg], [s / leg, c * leg]])
        scale = np.eye(2) * np.sqrt(span + 1e-5)
        self.arm1 = scipy.stats.multivariate_normal(self.ctr, r1 @ scale @ scale @ r1.T)
        self.arm2 = scipy.stats.multivariate_normal(self.ctr, r2 @ scale @ scale @ r2.T)

    # log-likelihood of one observation
    def llik(self, x):
        return np.log((self.arm1.pdf(x) + self.arm2.pdf(x)) / 2.0 + 1e-5)

    # n random samples
    def sample(self, n):
        a1_samples = self.arm1.rvs(n)
        a2_samples = self.arm2.rvs(n)
        return np.where(np.random.uniform(0, 1, n) < 0.5, a1_samples.T, a2_samples.T).T

    # render the distribution pdf as if it were a PMF
    def draw(self):
        span = np.linspace(0, 16, 160)
        grid = np.array(np.meshgrid(span, span)).reshape(2, -1).T
        pdf = 0.5 * self.arm1.pdf(grid) + 0.5 * self.arm2.pdf(grid)
        show_pmf(pdf.reshape(160, 160).T)

The cell below displays an example of this new density overlayed on the map. 

In [ ]:
# Here is an example of the distribution
cf = CrossDistribution(ctr=(6,5), span=5.1, leg=3.0, angle=-30.)

# for drawing the distribution with the observation data
cf.draw()
ax = plt.gca()
ax.scatter(submarine_observations[:,0], submarine_observations[:,1], marker='o', color='w', s=2, label="Observations")
ax.set_xlim(0,16)
ax.set_ylim(0,16)
ax.legend()
plt.show()

Of course, the parameters are wrong here, so the distribution does not fit the observations, but you can see that the shape is more similar. 

### Task 3: Fitting parameters by hand

Try to tune by hand the parameters `ctr`, `span`, `leg` and `angle` to fit the data better. Store the resulting `CrossDistribution` density object in `cf`. 

*Hints:*

- No visible test here, but you should be able to verify visually that the density overlaps the points reasonably well.
- Thinking about the meaning of the parameters will make this faster!
    - `ctr` is the center of the cross
    - `span` is the spread of the cross
    - `leg` controls how narrow or wide each leg of the cross is
    - `angle` rotates the cross around

In [ ]:
# now replace cf with a cross distribution that fits the data better by setting the 
# YOUR CODE HERE
raise NotImplementedError()


cf.draw()
ax = plt.gca()
ax.scatter(submarine_observations[:,0], submarine_observations[:,1], marker='o', color='w', s=2, label="Observations")
ax.set_xlim(0,16)
ax.set_ylim(0,16)
ax.legend()
plt.show()


### Task 4: Expectation

The time required to search a square depending on its distance to the base station is given by the formula $t = d^2 + 3d + 2$ (the same as last week). 

We would like to be able to compute the expected location (and search time/distance) for this model. But we can't do a simple operation with a PDF like we could with the PMF model: Because $X$ is a *continuous* random variable, the expectation is given by the integral

$$
\mathbb{E}[g(X)] = \int_x g(x) f_X(x) dx
$$

But this integral is not easily computable for the cross density $f^+_X(x)$! Instead, we can use sampling methods to approximate the expectation $\hat{\mathbb{E}}[f(X)]$. 

The *Monte Carlo approximation* of the expectation is given by: 

$$
\mathbb{E}[f(X)] \simeq \hat{\mathbb{E}}[f(X)] = \frac{1}{N}\sum_{i=0}^N g(x_i) 
$$
with all $x_i$ drawn from the density $f^+_X(x)$. 

In this task you are asked to use the *Monte Carlo approximation* to the expectation, with 1000 samples, to compute:

* `cross_expected_location` the expected location  (a 2D vector)
* `cross_expected_distance` expected L2 distance from the fixed station at (2,5) 
* `cross_expected_time`  expected search time in hours from this station.

with a cross distribution having the parameters:
    
        ctr = (4,6)
        span = 0.4
        leg = 5.0
        angle = 30.0

In [ ]:
# the position of the base station in the grid 
base = np.array((2,5))

# the function for the time to search a square at a given distance from the base station
def time(distance):
    return distance ** 2 + 3 * distance + 2

In [ ]:
# We set the seed for reproducible results - DO NOT CHANGE
np.random.seed(42) 
##########################
    
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
with tick.marks(1):
    assert check_scalar(np.sum(cross_expected_location), '0xb98735a1', tol=1)


In [ ]:
with tick.marks(1):
    assert check_scalar(cross_expected_distance, '0x7028ea0a', tol=1)

In [ ]:
# Hidden test cross_expected_time [1 mark]

## Part 3 - Estimating parameters by optimising the likelihood 
*Note* this is the difficult and challenging part of this lab.

There is no direct form to compute the parameters  `ctr`, `span`, `leg` and `angle` from `submarine_observations` in the way we could for the simple normal distribution. 

We can instead exploit the idea of maximum likelihood which aims to make inferences about the parameters that where most likely to generate the observations. Recall that we have the likelihood of a single observation under the model

$$\mathcal L(\theta;\mathbf{x}) = f_X(\mathbf{x};\theta)$$

where $\theta$ are the parameters `ctr`, `span`, `leg` and `angle` of our cross distribution and $\mathbf{x}$ is one of the 2D observations in `submarine_observations`.

If we assume our observations are independent and identically distributed, we can define an **objective function** for all observations, namely the product of the individual likelihood terms

$$\mathcal{L}(\theta ;{\{ {x_i}\} ^{i = 1:n}}) = -\prod\limits_{i = 1}^n {\mathcal{L}(\theta ;{x_i})} $$

If we take the log (perhaps to achieve greater numerical stability) and futher more take the negative value (as maximising the likelihood, is equivalent to minimiseing the negative likelihood), we get the following optimisation problem:

$$\theta^* = \arg\min_\theta L(\theta)$$
$$L(\theta) = -\log \mathcal{L}(\theta | x_1, \dots, x_n) = -\sum_i \log f_X(x_i;\theta) , $$

This is *very* similar to the approximation objective function we saw in the optimisation part of this course $\|f(\mathbf{x};\theta)-y\|$, but we have $y=0$ and we only have a scalar output from $f$ so the norm is unnecessary. We already know how to solve this kind of problem; just optimise. This is called **maximum likelihood estimation** and is a general technique for determining parameters of a distribution which we don't know given some observations. It will find the **best** setting of parameters that would explain how the observations came to be.



Using your own custom  **stochastic hill-climbing** (you can find code in the optimisation lab) and the idea of **maximum likelihood**, create a function `optimise_llik` to find the best parameter settings `theta= [ctr_x, ctr_y, span, leg, angle]` given the observations in `submarine_observations`. The signature of the function is given in the cell below.

In the specific setting we also know that; 

* `ctr` must be a vector in [0,0] to [16,16], 
* `span` must be a scalar in range 0.0 to 20.0
* `leg` must be a scalar from 0.0 to 10.0
* `angle` must be a scalar from 0.0 to 45.0 

which must be accounted for in your optimisation routine to find a best-fit estimate for `theta`. **Solutions not obeying the constraints will result in zero marks.**

You should be able to find a good fit within 10,000 iterations (max 15000). You will need to modify the stochastic hill climbing code to enforce the above constraints and ensure that you algorithm converges to a sensible solution by inspecting the plots and parameters. You may need to change the random seed and various parameters to arrive at a sensible solution.

**Note: We will test your solution on multiple instances so you cannot hardcode the solution. We will NOT accept solutions using any in-built optimization functions.**


### Task 5: 

First implement a function for the calculating the log-likelihood loss $L(\theta)$ as the function `cross_llik` below
The function's input are: 

- `theta` the parameters of the distribution we want to optimise
- `data` an array of observations, where each row provides the $x$ and $y$ coordinates of one observation. 
- `pdf` a python class for the chosen model density, in our case `CrossDistribution`
  
and its output should be the negative log likelihood $L(\theta)$ for `theta` and `data` 


In [ ]:
def cross_nllik(theta, data, pdf):
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:

# Sanity check of your function
with tick.marks(1): 
    assert cross_nllik([0.,0., 0., 0., 0.], submarine_observations, CrossDistribution) > 0.0, "ERROR: Negative log likelihood should return a positive number" 
    assert cross_nllik([0.,0., 0., 0., 0.], submarine_observations[:3,:], CrossDistribution) - cross_nllik([0.,0., 0., 0., 0.], submarine_observations[:2,:], CrossDistribution) > 0.0, "ERROR: Negative log likelihood should increase as you add observations"
    

### Task 6: 

Second, implement the guess function for the initial guess ($\theta_0$) of the hill climbing. 
You can assume a uniform distribution over the parameter range, for example using `np.random.uniform`. 

As a hint, you are provided with the function `invalid` to test whether a set of parameters are valid or not. 

In [ ]:
# This function tests whether theta values are within the allowed range
def invalid(theta):
    if not (0 <= theta[0] <= 16):
        return True
    if not (0 <= theta[1] <= 16):
        return True
    if not (0.0 <= theta[2] <= 20):
        return True
    if not (0.0 <= theta[3] <= 10):
        return True
    if not (0 <= theta[4] <= 45.0):
        return True
    return False

In [ ]:
def guess():
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
# Sanity check of your function
with tick.marks(1): 
    tests = [guess() for _ in range(1000)]
    for t in tests: 
        assert not invalid(t), f"ERROR: theta={t} is invalid!"

### Task 7: Neighbour function 

Third, implement a neighbour function that will provide a *candidate* solution $\theta'$ from your current best solution $\theta$. 
You can use `np.random.normal` to sample each value of $\theta'$ from a unimodal Normal distribution centred on the corresponding parameter in $\theta$ with a variance of your choice. 


In [ ]:
def neighbour(theta):
    # YOUR CODE HERE
    raise NotImplementedError()

You are then provided with the hill climbing code similar from what we did in part 2 of this course. 

In [ ]:
def optimise_llik(observations, pdf, max_iters, guess_fn, neighbour_fn):
    """
        observations: a numpy array with the observations (e.g submarine_observations)
        pdf: the PDF (e.g. CrossDistribution)
        iters: number of iterations   
        
        Note: you would likely need to implement a number of helper functions in the cell above.
    """

    best_guess = guess_fn()
    best_loss = cross_nllik(best_guess, observations, pdf)  # work out how bad it is

    for i in range(max_iters):
        if i % 100 == 0:
            print(i, best_guess, best_loss)  
            
        proposed = neighbour_fn(best_guess)
        if not invalid(proposed): 
            try: 
                proposed_loss = cross_nllik(proposed, observations, pdf)
            except: 
                print(proposed)
                raise
                
            # check if we beat the record
            if proposed_loss < best_loss:
                best_guess, best_loss = proposed, proposed_loss
    
    return best_guess # return the found parameters


The following cell will run the hill climbing optimisation. The whole 5000 iterations should only take a minute to run. 

**If it takes significantly longer, there is something wrong in your code!**

In [ ]:
iters = 5000
np.random.seed(42) 
theta = optimise_llik(observations=submarine_observations, pdf=CrossDistribution, max_iters=iters, guess_fn=guess, neighbour_fn=neighbour)
print("Estimated parameters:", theta)


The cell below displays the resulting density overlayed on the map and the observations. They should be a good fit, if not try and tweak your neighbour function by changing the variance. 

In [ ]:
cf = CrossDistribution(ctr=(theta[0],theta[1]), span=theta[2], leg=theta[3], angle=theta[4])
cf.draw()
ax = plt.gca()
ax.scatter(submarine_observations[:,0], submarine_observations[:,1], marker='o', color='w', s=2, label="Observations")
ax.set_xlim(0,16)
ax.set_ylim(0,16)
fig_2_5 = plt.gcf() # do not modify
plt.show()

In [ ]:
# Hidden auto graded [2 marks] - if your optimiser gets you in the vicinity of the true parameters 

In [ ]:
# Hidden auto graded [2 marks]... if your optimiser gets you very close to the true parameters

---
## Before you finish

1. **Kernel &rarr; Restart Kernel and Run All Cells.** The notebook must run top to bottom with no errors (no red boxes, no tracebacks).
2. Check the **marks summary** below.
3. Upload the completed `.ipynb` to Moodle. 



# Appendix: Marks summary

Run **Kernel &rarr; Restart Kernel and Run All Cells** before reading this, or it will be out of date.


In [ ]:
print("Marks available: 20  (visible with feedback: 10,  hidden: 10)")

# tick.summarise_marks() only reads the id-based tick.Marks() records, which this
# lab does not use; summarise from the counters tick.marks() maintains instead.
import IPython.display
if tick.available_marks > 0:
    IPython.display.display(IPython.display.HTML(
        """<!--{id:"TOTALMARK",marks:"%d", available:"%d"}  -->
        <div class="alert alert-box alert-success">
        <h1> Marks in this run: %d/%d (%.1f%%) </h1> </div>"""
        % (tick.total_marks, tick.available_marks,
           tick.total_marks, tick.available_marks,
           tick.total_marks / tick.available_marks * 100)))
else:
    print("No marks recorded -- use Kernel -> Restart Kernel and Run All Cells.")
